In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn
import pandas as pd
from google.colab import drive
from sklearn.model_selection import train_test_split
import shap

In [2]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# Загрузка данных

In [3]:
drive.mount("/content/drive")
!unzip /content/drive/MyDrive/credit_scoring_data/home-credit-default-risk.zip -d /content/data

Mounted at /content/drive
Archive:  /content/drive/MyDrive/credit_scoring_data/home-credit-default-risk.zip
  inflating: /content/data/HomeCredit_columns_description.csv  
  inflating: /content/data/POS_CASH_balance.csv  
  inflating: /content/data/application_test.csv  
  inflating: /content/data/application_train.csv  
  inflating: /content/data/bureau.csv  
  inflating: /content/data/bureau_balance.csv  
  inflating: /content/data/credit_card_balance.csv  
  inflating: /content/data/installments_payments.csv  
  inflating: /content/data/previous_application.csv  
  inflating: /content/data/sample_submission.csv  


In [4]:
app = pd.read_csv('/content/data/application_train.csv')
bureau = pd.read_csv('/content/data/bureau.csv')
bur_balance = pd.read_csv('/content/data/bureau_balance.csv')
credit_card_balance = pd.read_csv('/content/data/credit_card_balance.csv')
installments_payments = pd.read_csv('/content/data/installments_payments.csv')
prev_app = pd.read_csv('/content/data/previous_application.csv')
pos_cash_balance = pd.read_csv('/content/data/POS_CASH_balance.csv')



In [ ]:
app.shape

(307511, 122)

# Формирование витрины данных

In [5]:
application = app.copy()
if "DAYS_EMPLOYED" in application.columns:
    application["DAYS_EMPLOYED"] = application["DAYS_EMPLOYED"].replace(365243, np.nan)

if {"AMT_CREDIT", "AMT_INCOME_TOTAL"}.issubset(application.columns):
    application["CREDIT_TO_INCOME"] = (
        application["AMT_CREDIT"] / application["AMT_INCOME_TOTAL"].replace(0, np.nan)
    )

if {"AMT_ANNUITY", "AMT_INCOME_TOTAL"}.issubset(application.columns):
    application["ANNUITY_TO_INCOME"] = (
        application["AMT_ANNUITY"] / application["AMT_INCOME_TOTAL"].replace(0, np.nan)
    )

In [6]:
bureau_agg = (
    bureau.groupby("SK_ID_CURR")
    .agg(
        BUREAU_CNT=("SK_ID_BUREAU", "count"),
        BUREAU_DAYS_CREDIT_MAX=("DAYS_CREDIT", "max"),
        BUREAU_DAYS_CREDIT_AVG=("DAYS_CREDIT", "mean"),
        BUREAU_CREDIT_DAY_OVERDUE_MAX=("CREDIT_DAY_OVERDUE", "max"),
        BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM=("AMT_CREDIT_SUM_OVERDUE", "sum"),
        BUREAU_CNT_CREDIT_PROLONG_SUM=("CNT_CREDIT_PROLONG", "sum"),
        BUREAU_AMT_CREDIT_SUM_DEBT_MAX=("AMT_CREDIT_SUM_DEBT", "max"),
        BUREAU_AMT_CREDIT_SUM_DEBT_SUM=("AMT_CREDIT_SUM_DEBT", "sum"),
        BUREAU_AMT_CREDIT_SUM_SUM=("AMT_CREDIT_SUM", "sum"),
    )
    .reset_index()
)

status_cnt = pd.crosstab(bureau["SK_ID_CURR"], bureau["CREDIT_ACTIVE"]).reset_index()
status_cnt = status_cnt.rename(
    columns={
        "Active": "BUREAU_ACTIVE_CNT",
        "Closed": "BUREAU_CLOSED_CNT",
        "Sold": "BUREAU_SOLD_CNT",
        "Bad debt": "BUREAU_BAD_DEBT_CNT",
    }
)

for col in ["BUREAU_ACTIVE_CNT", "BUREAU_CLOSED_CNT", "BUREAU_SOLD_CNT", "BUREAU_BAD_DEBT_CNT"]:
    if col not in status_cnt.columns:
        status_cnt[col] = 0

bureau_agg = bureau_agg.merge(status_cnt, on="SK_ID_CURR", how="left")

In [7]:
prev = prev_app.copy()

if {"AMT_CREDIT", "AMT_APPLICATION"}.issubset(prev.columns):
    prev["CREDIT_APP_RATIO"] = prev["AMT_CREDIT"] / prev["AMT_APPLICATION"].replace(0, np.nan)

if {"AMT_ANNUITY", "AMT_CREDIT"}.issubset(prev.columns):
    prev["ANNUITY_CREDIT_RATIO"] = prev["AMT_ANNUITY"] / prev["AMT_CREDIT"].replace(0, np.nan)

prev_app_agg = (
    prev.groupby("SK_ID_CURR")
    .agg(
        PREV_APP_CNT=("SK_ID_PREV", "count"),
        PREV_APP_AMT_CREDIT_SUM=("AMT_CREDIT", "sum"),
        PREV_APP_AMT_CREDIT_MAX=("AMT_CREDIT", "max"),
        PREV_APP_CREDIT_APP_RATIO_AVG=("CREDIT_APP_RATIO", "mean"),
        PREV_APP_ANNUITY_CREDIT_RATIO_AVG=("ANNUITY_CREDIT_RATIO", "mean"),
        PREV_APP_CNT_PAYMENT_AVG=("CNT_PAYMENT", "mean"),
        PREV_APP_LAST_DAYS_DECISION=("DAYS_DECISION", "max"),
        PREV_APP_ANNUITY_CREDIT_AVG=("AMT_ANNUITY", "mean"),
    )
    .reset_index()
)

st = pd.crosstab(prev["SK_ID_CURR"], prev["NAME_CONTRACT_STATUS"])
st = st.rename(
    columns={
        "Approved": "PREV_APP_APPROVED_CNT",
        "Refused": "PREV_APP_REFUSED_CNT",
        "Canceled": "PREV_APP_CANCELED_CNT",
        "Unused offer": "PREV_APP_UNUSED_OFFER_CNT",
    }
)

for col in [
    "PREV_APP_APPROVED_CNT",
    "PREV_APP_REFUSED_CNT",
    "PREV_APP_CANCELED_CNT",
    "PREV_APP_UNUSED_OFFER_CNT",
]:
    if col not in st.columns:
        st[col] = 0

st["PREV_APP_TOTAL_CNT"] = st[
    ["PREV_APP_APPROVED_CNT", "PREV_APP_REFUSED_CNT", "PREV_APP_CANCELED_CNT", "PREV_APP_UNUSED_OFFER_CNT"]
].sum(axis=1)

denom = st["PREV_APP_TOTAL_CNT"].replace(0, np.nan)
st["PREV_APP_APPROVED_RATE"] = st["PREV_APP_APPROVED_CNT"] / denom
st["PREV_APP_REFUSED_RATE"] = st["PREV_APP_REFUSED_CNT"] / denom

st = st.reset_index()[["SK_ID_CURR", "PREV_APP_APPROVED_RATE", "PREV_APP_REFUSED_RATE"]]

In [62]:
df_final = (
    application
    .merge(bureau_agg, on="SK_ID_CURR", how="left")
    .merge(prev_app_agg, on="SK_ID_CURR", how="left")
    .merge(st, on="SK_ID_CURR", how="left")
)

df_final.shape
df_final, df_final_val = train_test_split(df_final, test_size=0.2, random_state=42, stratify=df_final["TARGET"])

In [63]:
df_final.columns.tolist()

['SK_ID_CURR',
 'TARGET',
 'NAME_CONTRACT_TYPE',
 'CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'CNT_CHILDREN',
 'AMT_INCOME_TOTAL',
 'AMT_CREDIT',
 'AMT_ANNUITY',
 'AMT_GOODS_PRICE',
 'NAME_TYPE_SUITE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'REGION_POPULATION_RELATIVE',
 'DAYS_BIRTH',
 'DAYS_EMPLOYED',
 'DAYS_REGISTRATION',
 'DAYS_ID_PUBLISH',
 'OWN_CAR_AGE',
 'FLAG_MOBIL',
 'FLAG_EMP_PHONE',
 'FLAG_WORK_PHONE',
 'FLAG_CONT_MOBILE',
 'FLAG_PHONE',
 'FLAG_EMAIL',
 'OCCUPATION_TYPE',
 'CNT_FAM_MEMBERS',
 'REGION_RATING_CLIENT',
 'REGION_RATING_CLIENT_W_CITY',
 'WEEKDAY_APPR_PROCESS_START',
 'HOUR_APPR_PROCESS_START',
 'REG_REGION_NOT_LIVE_REGION',
 'REG_REGION_NOT_WORK_REGION',
 'LIVE_REGION_NOT_WORK_REGION',
 'REG_CITY_NOT_LIVE_CITY',
 'REG_CITY_NOT_WORK_CITY',
 'LIVE_CITY_NOT_WORK_CITY',
 'ORGANIZATION_TYPE',
 'EXT_SOURCE_1',
 'EXT_SOURCE_2',
 'EXT_SOURCE_3',
 'APARTMENTS_AVG',
 'BASEMENTAREA_AVG',
 'YEARS_BEGINEXPLUATATION_A

# Предобработка данных

In [67]:
corr_matrix = df_final.corr(numeric_only=True)
corr_pairs = (
    corr_matrix
    .abs()
    .unstack()
    .sort_values(ascending=False)
)

corr_pairs = corr_pairs[corr_pairs < 1]
corr_pairs[corr_pairs > 0.6]

,,0
OBS_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,0.998514
OBS_60_CNT_SOCIAL_CIRCLE,OBS_30_CNT_SOCIAL_CIRCLE,0.998514
YEARS_BUILD_AVG,YEARS_BUILD_MEDI,0.998391
YEARS_BUILD_MEDI,YEARS_BUILD_AVG,0.998391
FLOORSMIN_MEDI,FLOORSMIN_AVG,0.997322
FLOORSMIN_AVG,FLOORSMIN_MEDI,0.997322
FLOORSMAX_AVG,FLOORSMAX_MEDI,0.996983
FLOORSMAX_MEDI,FLOORSMAX_AVG,0.996983
ENTRANCES_MEDI,ENTRANCES_AVG,0.996911
ENTRANCES_AVG,ENTRANCES_MEDI,0.996911


In [68]:
cols_to_drop = [

    # MEDI / MODE дубли
    "YEARS_BUILD_MEDI", "YEARS_BUILD_MODE",
    "FLOORSMIN_MEDI", "FLOORSMIN_MODE",
    "FLOORSMAX_MEDI", "FLOORSMAX_MODE",
    "ENTRANCES_MEDI", "ENTRANCES_MODE",
    "ELEVATORS_MEDI", "ELEVATORS_MODE",
    "COMMONAREA_MEDI", "COMMONAREA_MODE",
    "LIVINGAREA_MEDI", "LIVINGAREA_MODE",
    "APARTMENTS_MEDI", "APARTMENTS_MODE",
    "BASEMENTAREA_MEDI", "BASEMENTAREA_MODE",
    "LIVINGAPARTMENTS_MEDI", "LIVINGAPARTMENTS_MODE",
    "LANDAREA_MEDI", "LANDAREA_MODE",
    "NONLIVINGAPARTMENTS_MEDI", "NONLIVINGAPARTMENTS_MODE",
    "NONLIVINGAREA_MEDI", "NONLIVINGAREA_MODE",
    "YEARS_BEGINEXPLUATATION_MEDI", "YEARS_BEGINEXPLUATATION_MODE",

    # Соц. окружение
    "OBS_60_CNT_SOCIAL_CIRCLE",
    "DEF_60_CNT_SOCIAL_CIRCLE",

    # Регион
    "REGION_RATING_CLIENT_W_CITY",

    # Bureau
    "BUREAU_AMT_CREDIT_SUM_DEBT_MAX",
    "BUREAU_CLOSED_CNT",

    # Семья
    "CNT_CHILDREN",

    # Финансовые
    "AMT_GOODS_PRICE",

    # Previous application
    "PREV_APP_AMT_CREDIT_MAX",
    "APARTMENTS_AVG",
    "LIVINGAPARTMENTS_AVG",
    "TOTALAREA_MODE",
    "ELEVATORS_AVG",

    # Региональные дубли
    "LIVE_REGION_NOT_WORK_REGION",
    "LIVE_CITY_NOT_WORK_CITY",

    # Финансовые ratio (оставляем CREDIT_TO_INCOME)
    "ANNUITY_TO_INCOME",
]

df_final = df_final.drop(columns=cols_to_drop, errors="ignore")

In [70]:
corr_matrix = df_final.corr(numeric_only=True)
corr_pairs = (
    corr_matrix
    .abs()
    .unstack()
    .sort_values(ascending=False)
)

corr_pairs = corr_pairs[corr_pairs < 1]
corr_pairs[corr_pairs > 0.7]

,,0
AMT_CREDIT,AMT_ANNUITY,0.770163
AMT_ANNUITY,AMT_CREDIT,0.770163
FLOORSMIN_AVG,FLOORSMAX_AVG,0.739772
FLOORSMAX_AVG,FLOORSMIN_AVG,0.739772


In [71]:
df_final.isna().mean().sort_values(ascending=False)

,0
COMMONAREA_AVG,0.698396
NONLIVINGAPARTMENTS_AVG,0.693998
FONDKAPREMONT_MODE,0.683779
FLOORSMIN_AVG,0.678519
YEARS_BUILD_AVG,0.664787
OWN_CAR_AGE,0.660214
LANDAREA_AVG,0.593416
BASEMENTAREA_AVG,0.584652
EXT_SOURCE_1,0.563376
NONLIVINGAREA_AVG,0.551299


In [72]:
missing_ratio = df_final.isna().mean()

cols_to_drop = missing_ratio[missing_ratio > 0.60].index.tolist()

df_final = df_final.drop(columns=cols_to_drop)

print("Удалено признаков:", len(cols_to_drop))
print(cols_to_drop)

Удалено признаков: 6
['OWN_CAR_AGE', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'FLOORSMIN_AVG', 'NONLIVINGAPARTMENTS_AVG', 'FONDKAPREMONT_MODE']


In [73]:
df_final = df_final.drop(columns=["CODE_GENDER"]) # удаляем, так как это дискриминация по половому признаку

In [74]:
df_final.columns.tolist()

['SK_ID_CURR',
 'TARGET',
 'NAME_CONTRACT_TYPE',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'AMT_INCOME_TOTAL',
 'AMT_CREDIT',
 'AMT_ANNUITY',
 'NAME_TYPE_SUITE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'REGION_POPULATION_RELATIVE',
 'DAYS_BIRTH',
 'DAYS_EMPLOYED',
 'DAYS_REGISTRATION',
 'DAYS_ID_PUBLISH',
 'FLAG_MOBIL',
 'FLAG_EMP_PHONE',
 'FLAG_WORK_PHONE',
 'FLAG_CONT_MOBILE',
 'FLAG_PHONE',
 'FLAG_EMAIL',
 'OCCUPATION_TYPE',
 'CNT_FAM_MEMBERS',
 'REGION_RATING_CLIENT',
 'WEEKDAY_APPR_PROCESS_START',
 'HOUR_APPR_PROCESS_START',
 'REG_REGION_NOT_LIVE_REGION',
 'REG_REGION_NOT_WORK_REGION',
 'REG_CITY_NOT_LIVE_CITY',
 'REG_CITY_NOT_WORK_CITY',
 'ORGANIZATION_TYPE',
 'EXT_SOURCE_1',
 'EXT_SOURCE_2',
 'EXT_SOURCE_3',
 'BASEMENTAREA_AVG',
 'YEARS_BEGINEXPLUATATION_AVG',
 'ENTRANCES_AVG',
 'FLOORSMAX_AVG',
 'LANDAREA_AVG',
 'LIVINGAREA_AVG',
 'NONLIVINGAREA_AVG',
 'HOUSETYPE_MODE',
 'WALLSMATERIAL_MODE',
 'EMERGENCYSTATE_MODE',
 'OBS_30_CNT_SO

In [75]:
df_final["TARGET"].mean() # доля дефолтов в датасете

np.float64(0.08072908198107379)

In [76]:
def is_binary_numeric(series):                   # проверка, какие признаки бинарные
    unique_vals = set(series.dropna().unique())
    return unique_vals.issubset({0, 1})
for col in df_final.columns:
  if is_binary_numeric(df_final[col]):
    print(col)

TARGET
FLAG_MOBIL
FLAG_EMP_PHONE
FLAG_WORK_PHONE
FLAG_CONT_MOBILE
FLAG_PHONE
FLAG_EMAIL
REG_REGION_NOT_LIVE_REGION
REG_REGION_NOT_WORK_REGION
REG_CITY_NOT_LIVE_CITY
REG_CITY_NOT_WORK_CITY
FLAG_DOCUMENT_2
FLAG_DOCUMENT_3
FLAG_DOCUMENT_4
FLAG_DOCUMENT_5
FLAG_DOCUMENT_6
FLAG_DOCUMENT_7
FLAG_DOCUMENT_8
FLAG_DOCUMENT_9
FLAG_DOCUMENT_10
FLAG_DOCUMENT_11
FLAG_DOCUMENT_12
FLAG_DOCUMENT_13
FLAG_DOCUMENT_14
FLAG_DOCUMENT_15
FLAG_DOCUMENT_16
FLAG_DOCUMENT_17
FLAG_DOCUMENT_18
FLAG_DOCUMENT_19
FLAG_DOCUMENT_20
FLAG_DOCUMENT_21
BUREAU_BAD_DEBT_CNT


# Биннинг, WOE encoding и подсчет IV для каждого признака

In [77]:
df_work = df_final.copy()
target = "TARGET"

# drop id
if "SK_ID_CURR" in df_work.columns:
    df_work = df_work.drop(columns=["SK_ID_CURR"])

binary_cols = [c for c in df_work.columns if is_binary_numeric(df_work[c]) and c != target]


numeric_cols = df_work.select_dtypes(include=[np.number]).columns.tolist()
if target in numeric_cols:
    numeric_cols.remove(target)

categorical_cols = df_work.select_dtypes(include=["object"]).columns.tolist()

numeric_cols = [c for c in numeric_cols if c not in binary_cols]
categorical_cols = categorical_cols + binary_cols

categorical_cols = list(dict.fromkeys(categorical_cols))

def quantile_binning(series, q=10):
    s = series.copy()
    mask_na = s.isna()
    try:
        binned = pd.qcut(s[~mask_na], q=q, duplicates="drop")
    except Exception:
        binned = pd.cut(s[~mask_na], bins=q)
    out = pd.Series(index=s.index, dtype="object")
    out.loc[~mask_na] = binned.astype(str)
    out.loc[mask_na] = "__MISSING__"
    return out

binned_df = df_work.copy()

for col in numeric_cols:
    binned_df[col] = quantile_binning(df_work[col], q=10)

# категориальные + бинарные оставляем, но NaN пометим
for col in categorical_cols:
    binned_df[col] = binned_df[col].astype("object").where(binned_df[col].notna(), "__MISSING__")

# WOE/IV
def compute_woe_iv(df, feature, target):
    eps = 1e-6
    g = df.groupby(feature, dropna=False)[target].agg(["count", "sum"])
    g.columns = ["total", "bad"]
    g["good"] = g["total"] - g["bad"]

    total_good = g["good"].sum()
    total_bad = g["bad"].sum()

    g["dist_good"] = g["good"] / (total_good + eps)
    g["dist_bad"] = g["bad"] / (total_bad + eps)

    g["WOE"] = np.log((g["dist_good"] + eps) / (g["dist_bad"] + eps))
    g["IV"] = (g["dist_good"] - g["dist_bad"]) * g["WOE"]
    return g["WOE"].to_dict(), float(g["IV"].sum())

woe_df = pd.DataFrame(index=df_work.index)
iv_table = []

all_features = numeric_cols + categorical_cols

for col in all_features:
    woe_map, iv = compute_woe_iv(binned_df[[col, target]], col, target)
    iv_table.append({"feature": col, "IV": iv})
    woe_df[col] = binned_df[col].map(woe_map)

woe_df[target] = df_work[target]
iv_df = pd.DataFrame(iv_table).sort_values("IV", ascending=False)

iv_df.head(20)

,feature,IV
13,EXT_SOURCE_3,0.329764
12,EXT_SOURCE_2,0.303195
11,EXT_SOURCE_1,0.151344
33,BUREAU_DAYS_CREDIT_AVG,0.119970
5,DAYS_EMPLOYED,0.111552
4,DAYS_BIRTH,0.086772
32,BUREAU_DAYS_CREDIT_MAX,0.083580
58,OCCUPATION_TYPE,0.083309
60,ORGANIZATION_TYPE,0.075219
43,PREV_APP_CREDIT_APP_RATIO_AVG,0.070548


In [78]:
df_final = df_final.drop(columns=["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"], errors="ignore") # для честного скоринга не будем учитывать внешние скоринги, так как они отчасти дают "утечку" в данных
woe_df = woe_df.drop(columns=["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"], errors="ignore")
non_informative_features = iv_df[iv_df["IV"] < 0.015]                                               # удаляем неинформативные по IV колонки

df_final = df_final.drop(columns=non_informative_features["feature"].tolist(), errors="ignore")
woe_df = woe_df.drop(columns=non_informative_features["feature"].tolist(), errors="ignore")

In [79]:
woe_df.shape

(246008, 42)

# Считаем VIF для выявления мультиколлинеарности

In [80]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
X = woe_df.drop(columns=["TARGET"])
X = sm.add_constant(X)
vif_df = pd.DataFrame()
vif_df["features"] = X.columns
vif_df["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]
vif_df.sort_values("VIF", ascending=False).head(20)

,features,VIF
35,HOUSETYPE_MODE,8.760162
14,LIVINGAREA_AVG,8.272248
37,EMERGENCYSTATE_MODE,7.605976
11,ENTRANCES_AVG,7.011519
12,FLOORSMAX_AVG,6.506156
10,YEARS_BEGINEXPLUATATION_AVG,5.173881
36,WALLSMATERIAL_MODE,5.001079
15,NONLIVINGAREA_AVG,4.332797
9,BASEMENTAREA_AVG,4.283699
38,FLAG_EMP_PHONE,3.959695


In [81]:
high_vif = vif_df[vif_df["VIF"] > 5]
df_final = df_final.drop(columns=high_vif["features"], errors='ignore')
woe_df = woe_df.drop(columns=high_vif["features"], errors='ignore')
woe_df.shape

(246008, 35)

In [82]:
woe_df.columns.tolist()

['AMT_CREDIT',
 'AMT_ANNUITY',
 'REGION_POPULATION_RELATIVE',
 'DAYS_BIRTH',
 'DAYS_EMPLOYED',
 'DAYS_REGISTRATION',
 'DAYS_ID_PUBLISH',
 'REGION_RATING_CLIENT',
 'BASEMENTAREA_AVG',
 'LANDAREA_AVG',
 'NONLIVINGAREA_AVG',
 'DAYS_LAST_PHONE_CHANGE',
 'AMT_REQ_CREDIT_BUREAU_YEAR',
 'BUREAU_DAYS_CREDIT_MAX',
 'BUREAU_DAYS_CREDIT_AVG',
 'BUREAU_AMT_CREDIT_SUM_DEBT_SUM',
 'BUREAU_AMT_CREDIT_SUM_SUM',
 'BUREAU_ACTIVE_CNT',
 'PREV_APP_CREDIT_APP_RATIO_AVG',
 'PREV_APP_ANNUITY_CREDIT_RATIO_AVG',
 'PREV_APP_CNT_PAYMENT_AVG',
 'PREV_APP_ANNUITY_CREDIT_AVG',
 'PREV_APP_APPROVED_RATE',
 'PREV_APP_REFUSED_RATE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'OCCUPATION_TYPE',
 'ORGANIZATION_TYPE',
 'FLAG_EMP_PHONE',
 'REG_CITY_NOT_LIVE_CITY',
 'REG_CITY_NOT_WORK_CITY',
 'FLAG_DOCUMENT_3',
 'TARGET']

# Оцениваем ранжирующую способность признаков по коэффициенту Джини


In [83]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
def univariate_gini_table(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    rows = []

    for col in X_train.columns:
        s_tr = X_train[col].values
        s_te = X_test[col].values

        # если константа — пропускаем
        if np.std(s_tr) < 1e-12:
            continue

        auc_tr = roc_auc_score(y_train, s_tr)
        auc_te = roc_auc_score(y_test, s_te)

        rows.append({
            "feature": col,
            "gini_train": abs(2 * auc_tr - 1),
            "gini_test": abs(2 * auc_te - 1),
            "gini_drop": abs(2 * auc_tr - 1) - abs(2 * auc_te - 1)
        })

    return pd.DataFrame(rows).sort_values(
        "gini_train", ascending=False
    ).reset_index(drop=True)
gini_df = univariate_gini_table(woe_df.drop(columns=["TARGET"]), woe_df["TARGET"])
gini_df.head(20)

,feature,gini_train,gini_test,gini_drop
0,BUREAU_DAYS_CREDIT_AVG,0.197490,0.183996,0.013494
1,DAYS_EMPLOYED,0.187226,0.183818,0.003408
2,DAYS_BIRTH,0.169367,0.155598,0.013768
3,BUREAU_DAYS_CREDIT_MAX,0.163765,0.158832,0.004933
4,OCCUPATION_TYPE,0.152961,0.151887,0.001073
5,ORGANIZATION_TYPE,0.151233,0.150093,0.001140
6,PREV_APP_CREDIT_APP_RATIO_AVG,0.146750,0.151589,-0.004838
7,PREV_APP_APPROVED_RATE,0.127111,0.123549,0.003563
8,BUREAU_ACTIVE_CNT,0.124160,0.114397,0.009764
9,PREV_APP_REFUSED_RATE,0.122410,0.123027,-0.000617


In [84]:
low_gini = gini_df[gini_df['gini_train']<0.05]
df_final = df_final.drop(columns=low_gini["feature"], errors='ignore')
woe_df = woe_df.drop(columns=low_gini["feature"], errors='ignore')
woe_df.shape

(246008, 33)

In [85]:
woe_df.columns.tolist()

['AMT_CREDIT',
 'AMT_ANNUITY',
 'REGION_POPULATION_RELATIVE',
 'DAYS_BIRTH',
 'DAYS_EMPLOYED',
 'DAYS_REGISTRATION',
 'DAYS_ID_PUBLISH',
 'REGION_RATING_CLIENT',
 'BASEMENTAREA_AVG',
 'LANDAREA_AVG',
 'NONLIVINGAREA_AVG',
 'DAYS_LAST_PHONE_CHANGE',
 'AMT_REQ_CREDIT_BUREAU_YEAR',
 'BUREAU_DAYS_CREDIT_MAX',
 'BUREAU_DAYS_CREDIT_AVG',
 'BUREAU_AMT_CREDIT_SUM_DEBT_SUM',
 'BUREAU_AMT_CREDIT_SUM_SUM',
 'BUREAU_ACTIVE_CNT',
 'PREV_APP_CREDIT_APP_RATIO_AVG',
 'PREV_APP_ANNUITY_CREDIT_RATIO_AVG',
 'PREV_APP_CNT_PAYMENT_AVG',
 'PREV_APP_ANNUITY_CREDIT_AVG',
 'PREV_APP_APPROVED_RATE',
 'PREV_APP_REFUSED_RATE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'OCCUPATION_TYPE',
 'ORGANIZATION_TYPE',
 'FLAG_EMP_PHONE',
 'REG_CITY_NOT_WORK_CITY',
 'FLAG_DOCUMENT_3',
 'TARGET']

# Обучение логистической регрессии (для интерпретируемости)

In [86]:
FEATURES_33 = [
 'AMT_CREDIT','AMT_ANNUITY','REGION_POPULATION_RELATIVE','DAYS_BIRTH','DAYS_EMPLOYED',
 'DAYS_REGISTRATION','DAYS_ID_PUBLISH','REGION_RATING_CLIENT','BASEMENTAREA_AVG',
 'LANDAREA_AVG','NONLIVINGAREA_AVG','DAYS_LAST_PHONE_CHANGE','AMT_REQ_CREDIT_BUREAU_YEAR',
 'BUREAU_DAYS_CREDIT_MAX','BUREAU_DAYS_CREDIT_AVG','BUREAU_AMT_CREDIT_SUM_DEBT_SUM',
 'BUREAU_AMT_CREDIT_SUM_SUM','BUREAU_ACTIVE_CNT','PREV_APP_CREDIT_APP_RATIO_AVG',
 'PREV_APP_ANNUITY_CREDIT_RATIO_AVG','PREV_APP_CNT_PAYMENT_AVG','PREV_APP_ANNUITY_CREDIT_AVG',
 'PREV_APP_APPROVED_RATE','PREV_APP_REFUSED_RATE','NAME_INCOME_TYPE','NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS','OCCUPATION_TYPE','ORGANIZATION_TYPE','FLAG_EMP_PHONE',
 'REG_CITY_NOT_WORK_CITY','FLAG_DOCUMENT_3','TARGET'
]

def compute_woe_map(df_binned: pd.DataFrame, feature: str, target: str, eps: float = 1e-6):
    g = df_binned.groupby(feature, dropna=False)[target].agg(["count", "sum"])
    g.columns = ["total", "bad"]
    g["good"] = g["total"] - g["bad"]

    total_good = g["good"].sum()
    total_bad  = g["bad"].sum()

    g["dist_good"] = g["good"] / (total_good + eps)
    g["dist_bad"]  = g["bad"]  / (total_bad  + eps)

    woe = np.log((g["dist_good"] + eps) / (g["dist_bad"] + eps))
    return woe.to_dict()

def _is_binary_numeric(s: pd.Series) -> bool:
    if not pd.api.types.is_numeric_dtype(s):
        return False
    u = pd.Series(s.dropna().unique())
    return len(u) <= 2 and set(u.tolist()).issubset({0, 1})

def fit_woe_encoder(df_train: pd.DataFrame, features_33=FEATURES_33, target: str = "TARGET", q: int = 10):
    # 0) оставляем только нужные колонки (и target)
    cols = [c for c in features_33 if c in df_train.columns]
    df = df_train[cols].copy()

    # 1) типы
    feats = [c for c in cols if c != target]
    binary_cols = [c for c in feats if _is_binary_numeric(df[c])]
    numeric_cols = [c for c in feats if pd.api.types.is_numeric_dtype(df[c]) and c not in binary_cols]
    categorical_cols = [c for c in feats if c not in numeric_cols and c not in binary_cols] + binary_cols
    categorical_cols = list(dict.fromkeys(categorical_cols))

    # 2) fit "qcut edges" на train для numeric
    edges = {}
    for col in numeric_cols:
        s = df[col].astype(float)
        x = s.dropna().values
        if len(x) == 0:
            edges[col] = None
            continue
        qs = np.unique(np.nanquantile(x, np.linspace(0, 1, q + 1)))
        # если мало уникальных квантилей — биннинг в qcut развалится, fallback: равные bins по min/max
        if len(qs) < 3:
            mn, mx = np.nanmin(x), np.nanmax(x)
            if np.isfinite(mn) and np.isfinite(mx) and mn < mx:
                qs = np.linspace(mn, mx, q + 1)
            else:
                qs = None
        edges[col] = qs

    # 3) бинним train и считаем woe_map
    binned = pd.DataFrame(index=df.index)
    for col in numeric_cols:
        s = df[col].astype(float)
        mask_na = s.isna()
        out = pd.Series(index=df.index, dtype="object")
        if edges[col] is None:
            out.loc[~mask_na] = "__ALL__"
        else:
            out.loc[~mask_na] = pd.cut(s.loc[~mask_na], bins=edges[col], include_lowest=True).astype(str)
        out.loc[mask_na] = "__MISSING__"
        binned[col] = out

    for col in categorical_cols:
        binned[col] = df[col].astype("object").where(df[col].notna(), "__MISSING__")

    # 4) WOE maps (только по train)
    woe_maps = {}
    for col in (numeric_cols + categorical_cols):
        tmp = pd.DataFrame({col: binned[col], target: df[target].astype(int)})
        woe_maps[col] = compute_woe_map(tmp, col, target)

    meta = {
        "target": target,
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "edges": edges,
        "woe_maps": woe_maps,
        "features_order": feats,  # порядок фичей без target
    }
    return meta

def transform_woe(df_any: pd.DataFrame, meta: dict, features_33=FEATURES_33, neutral_woe: float = 0.0):
    target = meta["target"]
    cols = [c for c in features_33 if c in df_any.columns]
    df = df_any[cols].copy()

    # бинним по train-границам
    binned = pd.DataFrame(index=df.index)

    for col in meta["numeric_cols"]:
        if col not in df.columns:
            continue
        s = df[col].astype(float)
        mask_na = s.isna()
        out = pd.Series(index=df.index, dtype="object")
        ed = meta["edges"].get(col)
        if ed is None:
            out.loc[~mask_na] = "__ALL__"
        else:
            out.loc[~mask_na] = pd.cut(s.loc[~mask_na], bins=ed, include_lowest=True).astype(str)
        out.loc[mask_na] = "__MISSING__"
        binned[col] = out

    for col in meta["categorical_cols"]:
        if col not in df.columns:
            continue
        binned[col] = df[col].astype("object").where(df[col].notna(), "__MISSING__")

    # применяем WOE maps; unseen -> neutral_woe
    woe_df = pd.DataFrame(index=df.index)
    for col in meta["features_order"]:
        if col not in binned.columns:
            # если колонки нет — заполним нейтрально (или можно бросить ошибку)
            woe_df[col] = neutral_woe
            continue
        wmap = meta["woe_maps"].get(col, {})
        woe_df[col] = binned[col].map(wmap).fillna(neutral_woe)

    # добавить target если есть
    if target in df.columns:
        woe_df[target] = df[target].astype(int).values

    # привести к ровно 33 колонкам, как ты хочешь
    ordered = [c for c in FEATURES_33 if c != "TARGET"]
    woe_df = woe_df.reindex(columns=ordered + ([target] if target in woe_df.columns else []))
    return woe_df
meta = fit_woe_encoder(df_final, q=10)
woe_df_val = transform_woe(df_final_val, meta)


In [100]:
from sklearn.linear_model import LogisticRegression
X_train, y_train = woe_df.drop(columns=["TARGET"]), woe_df["TARGET"]
X_val, y_val = woe_df_val.drop(columns=["TARGET"]), woe_df_val["TARGET"]
model = LogisticRegression(penalty="l2", random_state=42)
model.fit(X_train, y_train)
proba_train = model.predict_proba(X_train)[:, 1]
proba_val = model.predict_proba(X_val)[:, 1]
auc_train = roc_auc_score(y_train, proba_train)
auc_val = roc_auc_score(y_val, proba_val)
print("AUC for train:", auc_train)
print("AUC for validation:", auc_val)

AUC for train: 0.7012653467617505
AUC for validation: 0.7017059955255789
KS: 0.29407249451455747


In [103]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.8 MB/s eta 0:00:00


In [105]:
from catboost import CatBoostClassifier, Pool
cat_model = CatBoostClassifier(iterations=1000,
                           learning_rate=0.05,
                           max_depth=6,
                           loss_function='Logloss',
                           early_stopping_rounds=200,
                           eval_metric='AUC',
                           verbose=200)
train_pool = Pool(X_train, y_train)
val_pool = Pool(X_val, y_val)
cat_model.fit(train_pool, eval_set=val_pool, use_best_model=True)


0:	test: 0.6102578	best: 0.6102578 (0)	total: 246ms	remaining: 4m 5s
200:	test: 0.7134013	best: 0.7134575 (197)	total: 18.1s	remaining: 1m 12s
400:	test: 0.7170931	best: 0.7171238 (398)	total: 35.6s	remaining: 53.2s
600:	test: 0.7182256	best: 0.7182256 (600)	total: 53.2s	remaining: 35.3s
800:	test: 0.7184703	best: 0.7187172 (763)	total: 1m 10s	remaining: 17.5s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.7187171889
bestIteration = 763

Shrink model to first 764 iterations.


CatBoostClassifier(early_stopping_rounds=200, eval_metric='AUC', iterations=1000, learning_rate=0.05, loss_function='Logloss', max_depth=6, verbose=200)